# Inspecting Circuits

To know the depth of the circuit, and try to avoid noise in real hardware and Barren Plateaus

In [ ]:
import pennylane as qml
from pennylane import numpy as np

dev_inspect = qml.device("default.qubit", wires=2)

@qml.qnode(dev_inspect)
def inspection_circuit(weights):
    # Some arbitrary gates for our model
    qml.RX(weights[0], wires=0)
    qml.RY(weights[1], wires=1)
    qml.CNOT(wires=[0, 1])
    qml.RX(weights[2], wires=0)
    
    return qml.expval(qml.PauliZ(0))

weights = np.array([0.1, 0.2, 0.3], requires_grad=True)


# CIRCUIT SPECIFICATIONS (Resource Estimation)

print("--- Circuit Specs ---")

circuit_specs = qml.specs(inspection_circuit)(weights)
resources = circuit_specs['resources']

print(f"Total operations (gates): {resources.num_gates}")
print(f"Circuit depth: {resources.depth}")
print(f"Device name: {circuit_specs['device_name']}\n")

print(f"Gate types used: {resources.gate_types}\n")


# 2. CIRCUIT MATRIX 

print("--- Full Unitary Matrix of the Circuit ---")
# Calculates the mathematical matrix representing the operations before measurement
# Shape will be (4, 4) since we have 2 qubits (2^2 = 4 basis states)
unitary_matrix = qml.matrix(inspection_circuit)(weights)
print(np.round(unitary_matrix, 2))

--- Circuit Specs ---
Total operations (gates): 4
Circuit depth: 3
Device name: default.qubit

Gate types used: {'RX': 2, 'RY': 1, 'CNOT': 1}

--- Full Unitary Matrix of the Circuit ---
[[ 0.98+0.j   -0.11+0.j    0.  -0.06j  0.  -0.14j]
 [ 0.09+0.j    0.98+0.j    0.  -0.15j  0.  -0.03j]
 [ 0.  -0.15j  0.  -0.03j  0.09+0.j    0.98+0.j  ]
 [ 0.  -0.06j  0.  -0.14j  0.98+0.j   -0.11+0.j  ]]


# Compiling Circuits

Optimize the circuit for the hardware

In [ ]:
import pennylane as qml
from pennylane import numpy as np

dev_compile = qml.device("default.qubit", wires=1)

# Define unoptimized quantum function
def unoptimized_ansatz(angle):
    # Two Hadamards in a row
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=0)
    
    # Two consecutive rotations on the same axis
    qml.RX(angle, wires=0)
    qml.RX(0.5, wires=0)
    
    return qml.expval(qml.PauliZ(0))

# Create QNode
unoptimized_qnode = qml.QNode(unoptimized_ansatz, dev_compile)
angle = 0.2


# BEFORE COMPILATION

print("--- ORIGINAL CIRCUIT ---")
print(qml.draw(unoptimized_qnode)(angle)) # Redundant gates

# Accessing depth via ['resources'].depth
original_depth = qml.specs(unoptimized_qnode)(angle)['resources'].depth
print(f"Original Depth: {original_depth}\n")


# AFTER COMPILATION

# Apply the compiler pipeline to optimize the QNode
# By default, it cancels inverses and merges rotations
compiled_qnode = qml.compile(unoptimized_qnode)

print("--- COMPILED CIRCUIT ---")
print(qml.draw(compiled_qnode)(angle)) # Hadamards disappeared and RX gates merged

# Accessing depth via ['resources'].depth
compiled_depth = qml.specs(compiled_qnode)(angle)['resources'].depth
print(f"Compiled Depth: {compiled_depth}")

--- ORIGINAL CIRCUIT ---
Notice the redundant gates:
0: ──H──H──RX(0.20)──RX(0.50)─┤  <Z>
Original Depth: 4

--- COMPILED CIRCUIT ---
Notice how Hadamards disappeared and RX gates merged (0.2 + 0.5 = 0.7):
0: ──RX(0.70)─┤  <Z>
Compiled Depth: 1
